# Evaluación multisemilla por ganancia de los mejores modelos de z495

Esta notebook:

1. carga el dataset crudo y genera `clase_ternaria`;
2. lee un `BO_log.txt` generado por R o por Optuna/Python;
3. selecciona los mejores `n` conjuntos de hiperparámetros;
4. entrena cada conjunto con `m` semillas distintas sobre todos los datos de entrenamiento;
5. calcula y promedia la ganancia sobre el período futuro, como en Producción de z494.

No ejecuta una nueva optimización bayesiana.


## Librerías

Si falta algún paquete, instalarlo previamente en el entorno de R correspondiente.


In [16]:
library(data.table)
library(here)
library(primes)
library(lightgbm)

options(scipen = 999)


## Configuración

`experimento_origen` puede ser el nombre de una carpeta dentro de `DATA/EXP` o una ruta completa.


In [17]:
root <- here()
dataset_folder <- file.path(root, "DATA", "DATASETS")
exp_folder <- file.path(root, "DATA", "EXP")

PARAM <- list()
PARAM$experimento_origen <- "PY495-02"
PARAM$archivo_resultados <- "BO_log.txt"

PARAM$semilla_primigenia <- 230047
PARAM$n_mejores <- 1L
PARAM$m_semillas <- 5L

PARAM$train_final <- c(202104)
PARAM$future <- c(202106)
PARAM$semilla_kaggle <- 314159
PARAM$cortes <- seq(4000, 19000, by = 500)
PARAM$trainingstrategy$undersampling <- 0.1


In [18]:
# Parámetros fijos de z495. Los presentes en BO_log.txt los pisan.
PARAM$lgbm$param_fijos <- list(
  boosting = "gbdt",
  objective = "binary",
  metric = "average_precision",
  feature_pre_filter = FALSE,
  first_metric_only = FALSE,
  boost_from_average = TRUE,
  force_row_wise = TRUE,
  deterministic = TRUE,
  verbosity = -100,

  seed = PARAM$semilla_primigenia,

  max_depth = -1L,
  min_gain_to_split = 0,
  lambda_l1 = 0.0,
  lambda_l2 = 0.0,
  max_bin = 31L,

  bagging_fraction = 1.0,
  pos_bagging_fraction = 1.0,
  neg_bagging_fraction = 1.0,
  is_unbalance = FALSE,
  scale_pos_weight = 1.0,

  drop_rate = 0.1,
  max_drop = 50L,
  skip_drop = 0.5,
  extra_trees = FALSE,

  early_stopping = 0L,
  min_data_in_leaf = 0L,
  feature_fraction = 0.50,
  learning_rate = 0.005,

  num_iterations = 20000L,
  num_leaves = 1024L,
  min_sum_hessian_in_leaf = 1e-6
)


## Dataset y clase ternaria

Se conserva la lógica de generación utilizada en z495.


In [19]:
dataset_url <- file.path(dataset_folder, "competencia_01_crudo.csv")

if (!file.exists(dataset_url)) {
  stop("No existe el dataset: ", dataset_url)
}

dataset <- fread(dataset_url)

dsimple <- dataset[, .(
  pos = .I,
  numero_de_cliente,
  periodo0 = as.integer(foto_mes / 100) * 12 + foto_mes %% 100
)]

setorder(dsimple, numero_de_cliente, periodo0)

periodo_ultimo <- dsimple[, max(periodo0)]
periodo_anteultimo <- periodo_ultimo - 1L

dsimple[, c("periodo1", "periodo2") :=
  shift(periodo0, n = 1:2, fill = NA, type = "lead"),
  by = numero_de_cliente
]

dsimple[periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA"]

dsimple[
  periodo0 < periodo_ultimo &
    (is.na(periodo1) | periodo0 + 1L < periodo1),
  clase_ternaria := "BAJA+1"
]

dsimple[
  periodo0 < periodo_anteultimo & periodo0 + 1L == periodo1 &
    (is.na(periodo2) | periodo0 + 2L < periodo2),
  clase_ternaria := "BAJA+2"
]

setorder(dsimple, pos)
dataset[, clase_ternaria := dsimple$clase_ternaria]

setorder(dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, by = .(foto_mes, clase_ternaria)]


foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202103,BAJA+1,1019
202103,BAJA+2,960
202103,CONTINUA,160921
202104,BAJA+1,964
202104,BAJA+2,1139
202104,CONTINUA,161181
202105,BAJA+1,1143
202105,BAJA+2,870
202105,CONTINUA,161755


In [20]:
dataset[, clase01 := fifelse(
  clase_ternaria %in% c("BAJA+2", "BAJA+1"),
  1L,
  0L
)]

dataset_train <- copy(dataset[foto_mes %in% PARAM$train_final])
dfuture <- copy(dataset[foto_mes %in% PARAM$future])

campos_buenos <- setdiff(
  names(dataset_train),
  c(
    "clase_ternaria", "clase01", "fold", "azar", "training",
    "cprestamos_personales", "mprestamos_personales"
  )
)

dtrain_final <- lgb.Dataset(
  data = data.matrix(dataset_train[, ..campos_buenos]),
  label = dataset_train[, clase01]
)

dataset_train[, .N, by = clase_ternaria]


clase_ternaria,N
<chr>,<int>
BAJA+1,964
BAJA+2,1139
CONTINUA,161181


## Lectura de resultados de R o Python

La función reconoce:

- `metrica` en el log de z495/R;
- `value` y columnas `params_*` en el log de Optuna/Python;
- `y` en otros logs de R compatibles.


In [21]:
resolver_carpeta_experimento <- function(experimento) {
  if (dir.exists(experimento)) {
    return(normalizePath(experimento, mustWork = TRUE))
  }

  candidato <- file.path(exp_folder, experimento)
  if (!dir.exists(candidato)) {
    stop("No existe la carpeta del experimento: ", candidato)
  }

  normalizePath(candidato, mustWork = TRUE)
}


leer_mejores_resultados <- function(carpeta, archivo, n_mejores) {
  ruta <- file.path(carpeta, archivo)
  if (!file.exists(ruta)) {
    stop("No existe el archivo de resultados: ", ruta)
  }

  tb <- fread(ruta)
  tb[, fila_origen := .I]

  if ("state" %in% names(tb)) {
    tb <- tb[state == "COMPLETE"]
  }

  # Optuna escribe los hiperparámetros como params_nombre.
  columnas_python <- grep("^params_", names(tb), value = TRUE)
  for (columna in columnas_python) {
    nombre_lgbm <- sub("^params_", "", columna)
    if (!nombre_lgbm %in% names(tb)) {
      setnames(tb, columna, nombre_lgbm)
    }
  }

  candidatas_metrica <- c("metrica", "value", "y")
  campo_metrica <- candidatas_metrica[candidatas_metrica %in% names(tb)][1]

  if (is.na(campo_metrica)) {
    stop("No se encontró una columna de métrica: metrica, value o y")
  }

  tb[, metrica_origen := as.numeric(get(campo_metrica))]
  tb <- tb[is.finite(metrica_origen)]

  if (nrow(tb) == 0L) {
    stop("El archivo no contiene evaluaciones completas con una métrica válida")
  }

  setorder(tb, -metrica_origen)
  tb <- head(tb, min(as.integer(n_mejores), nrow(tb)))
  tb[, ranking_origen := .I]

  if ("iter" %in% names(tb)) {
    tb[, id_origen := as.character(iter)]
  } else if ("number" %in% names(tb)) {
    tb[, id_origen := as.character(number)]
  } else {
    tb[, id_origen := as.character(fila_origen)]
  }

  tb[]
}


In [22]:
carpeta_experimento <- resolver_carpeta_experimento(
  PARAM$experimento_origen
)

mejores_modelos <- leer_mejores_resultados(
  carpeta = carpeta_experimento,
  archivo = PARAM$archivo_resultados,
  n_mejores = PARAM$n_mejores
)

columnas_parametros <- intersect(
  names(PARAM$lgbm$param_fijos),
  names(mejores_modelos)
)

columnas_mostrar <- unique(c(
  "ranking_origen", "id_origen", "metrica_origen",
  columnas_parametros
))

mejores_modelos[, ..columnas_mostrar]


ranking_origen,id_origen,metrica_origen,learning_rate,num_iterations,num_leaves,min_sum_hessian_in_leaf
<int>,<chr>,<dbl>,<dbl>,<int>,<int>,<dbl>
1,5,0.1182831,0.003697844,2780,24,0.06401048


## Semillas

Se generan semillas primas reproducibles y se utiliza `L'Ecuyer-CMRG`, siguiendo el manejo de semillas de las notebooks en R.


In [23]:
primos <- generate_primes(100000, 1000000)

if (PARAM$m_semillas > length(primos)) {
  stop("m_semillas supera la cantidad de números primos disponibles")
}

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
PARAM$semillas <- sample(
  primos,
  size = PARAM$m_semillas,
  replace = FALSE
)

PARAM$semillas


[1] 150497 931363 240707 993683 121349

## Ganancia de Producción de z494

Se conserva la división fija `Public`/`Private` de z494. La semilla Kaggle es independiente de las semillas usadas para entrenar los modelos.


In [24]:
particionar <- function(
  data,
  division,
  agrupa = "",
  campo = "fold",
  start = 1L,
  seed = NA
) {
  if (!is.na(seed)) {
    set.seed(seed, kind = "L'Ecuyer-CMRG")
  }

  bloque <- unlist(mapply(
    function(x, y) rep(y, x),
    division,
    seq(from = start, length.out = length(division))
  ))

  data[, (campo) := sample(
    rep(bloque, ceiling(.N / length(bloque)))
  )[1:.N], by = agrupa]
}


realidad_inicializar <- function(pfuture, pparam) {
  drealidad <- pfuture[, .(
    numero_de_cliente,
    foto_mes,
    clase_ternaria
  )]

  particionar(
    data = drealidad,
    division = c(3L, 7L),
    agrupa = "clase_ternaria",
    seed = pparam$semilla_kaggle
  )

  drealidad
}


realidad_evaluar <- function(prealidad, pprediccion) {
  prealidad[
    pprediccion,
    on = c("numero_de_cliente", "foto_mes"),
    predicted := i.Predicted
  ]

  tbl <- prealidad[, .(
    qty = .N
  ), by = .(fold, predicted, clase_ternaria)]

  res <- list(
    public = tbl[
      fold == 1L & predicted == 1L,
      sum(qty * fifelse(clase_ternaria == "BAJA+2", 1072500, -27500))
    ] / 0.3,
    private = tbl[
      fold == 2L & predicted == 1L,
      sum(qty * fifelse(clase_ternaria == "BAJA+2", 1072500, -27500))
    ] / 0.7,
    total = tbl[
      predicted == 1L,
      sum(qty * fifelse(clase_ternaria == "BAJA+2", 1072500, -27500))
    ]
  )

  prealidad[, predicted := NULL]
  res
}


drealidad <- realidad_inicializar(dfuture, PARAM)


## Entrenamiento y evaluación de los mejores modelos

Cada fila del log pisa los parámetros fijos correspondientes. Cada semilla cambia el entrenamiento de LightGBM, pero no los hiperparámetros del experimento.


In [25]:
extraer_parametros_modelo <- function(fila_modelo, semilla) {
  disponibles <- intersect(
    names(PARAM$lgbm$param_fijos),
    names(fila_modelo)
  )

  parametros_log <- as.list(fila_modelo[, ..disponibles])
  parametros_log <- parametros_log[vapply(
    parametros_log,
    function(valor) length(valor) == 1L && !is.na(valor),
    logical(1)
  )]

  parametros <- modifyList(
    PARAM$lgbm$param_fijos,
    parametros_log
  )

  parametros$seed <- as.integer(semilla)

  parametros_enteros <- intersect(
    c(
      "num_iterations", "num_leaves", "max_depth", "max_bin",
      "max_drop", "early_stopping", "min_data_in_leaf"
    ),
    names(parametros)
  )

  parametros[parametros_enteros] <- lapply(
    parametros[parametros_enteros],
    as.integer
  )

  # Normalización de Producción de z494. En z495 permanece en cero.
  parametros$min_data_in_leaf <- round(
    parametros$min_data_in_leaf /
      PARAM$trainingstrategy$undersampling
  )

  parametros
}


entrenar_y_predecir <- function(fila_modelo, semilla) {
  parametros <- extraer_parametros_modelo(fila_modelo, semilla)

  modelo <- lgb.train(
    data = dtrain_final,
    param = parametros
  )

  prediccion <- predict(
    modelo,
    data.matrix(dfuture[, ..campos_buenos])
  )

  rm(modelo)

  prediccion
}


In [26]:
resultados <- vector(
  mode = "list",
  length = nrow(mejores_modelos) *
    length(PARAM$semillas) *
    length(PARAM$cortes)
)

k <- 0L

for (modelo_id in seq_len(nrow(mejores_modelos))) {
  cat('nuevo modelo')
  flush.console()
  fila_modelo <- mejores_modelos[modelo_id]

  for (semilla in PARAM$semillas) {
    cat('nueva semilla')
    flush.console()
    prediccion <- entrenar_y_predecir(
      fila_modelo = fila_modelo,
      semilla = semilla
    )

    tb_prediccion <- dfuture[, .(
      numero_de_cliente,
      foto_mes,
      clase_ternaria
    )]

    tb_prediccion[, prob := prediccion]
    setorder(tb_prediccion, -prob)

    for (envios in PARAM$cortes) {
      k <- k + 1L

      tb_prediccion[, Predicted := 0L]
      tb_prediccion[seq_len(envios), Predicted := 1L]

      ganancia <- realidad_evaluar(
        prealidad = drealidad,
        pprediccion = tb_prediccion
      )

      resultados[[k]] <- data.table(
        modelo_id = modelo_id,
        ranking_origen = fila_modelo$ranking_origen,
        id_origen = fila_modelo$id_origen,
        metrica_origen = fila_modelo$metrica_origen,
        semilla = as.integer(semilla),
        envios = as.integer(envios),
        ganancia_total = ganancia$total,
        ganancia_public_estimada = ganancia$public,
        ganancia_private_estimada = ganancia$private
      )

      cat(
        "Modelo=", modelo_id,
        " Semilla=", semilla,
        " Envios=", envios,
        " TOTAL=", ganancia$total,
        " Public=", ganancia$public,
        " Private=", ganancia$private,
        "\n",
        sep = ""
      )
    }

    rm(prediccion, tb_prediccion)
    gc(full = TRUE, verbose = FALSE)
  }
}

resultados_detalle <- rbindlist(resultados)
resultados_detalle[]


Modelo=1 Semilla=150497 Envios=4000 TOTAL=315700000 Public=312583333 Private=317035714
Modelo=1 Semilla=150497 Envios=4500 TOTAL=336050000 Public=350441667 Private=329882143
Modelo=1 Semilla=150497 Envios=5000 TOTAL=349800000 Public=367858333 Private=342060714
Modelo=1 Semilla=150497 Envios=5500 TOTAL=355850000 Public=373083333 Private=348464286
Modelo=1 Semilla=150497 Envios=6000 TOTAL=366300000 Public=374458333 Private=362803571
Modelo=1 Semilla=150497 Envios=6500 TOTAL=376750000 Public=387750000 Private=372035714
Modelo=1 Semilla=150497 Envios=7000 TOTAL=392700000 Public=395908333 Private=391325000
Modelo=1 Semilla=150497 Envios=7500 TOTAL=399850000 Public=417450000 Private=392307143
Modelo=1 Semilla=150497 Envios=8000 TOTAL=405900000 Public=433583333 Private=394035714
Modelo=1 Semilla=150497 Envios=8500 TOTAL=414150000 Public=445316667 Private=400792857
Modelo=1 Semilla=150497 Envios=9000 TOTAL=418000000 Public=456225000 Private=401617857
Modelo=1 Semilla=150497 Envios=9500 TOTAL=4

modelo_id,ranking_origen,id_origen,metrica_origen,semilla,envios,ganancia_total,ganancia_public_estimada,ganancia_private_estimada
<int>,<int>,<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>
1,1,5,0.1182831,150497,4000,315700000,312583333,317035714
1,1,5,0.1182831,150497,4500,336050000,350441667,329882143
1,1,5,0.1182831,150497,5000,349800000,367858333,342060714
1,1,5,0.1182831,150497,5500,355850000,373083333,348464286
1,1,5,0.1182831,150497,6000,366300000,374458333,362803571
1,1,5,0.1182831,150497,6500,376750000,387750000,372035714
1,1,5,0.1182831,150497,7000,392700000,395908333,391325000
1,1,5,0.1182831,150497,7500,399850000,417450000,392307143
1,1,5,0.1182831,150497,8000,405900000,433583333,394035714


## Promedio de ganancias

`resumen_ganancias` muestra cada corte promediado sobre las `m` semillas. `mejor_corte_por_modelo` conserva el corte con mayor ganancia total promedio para cada conjunto de hiperparámetros.


In [27]:
resumen_metricas <- resultados_detalle[, .(
  ganancia_total_promedio = mean(ganancia_total),
  ganancia_total_sd = if (.N > 1L) sd(ganancia_total) else 0,
  ganancia_public_promedio = mean(ganancia_public_estimada),
  ganancia_private_promedio = mean(ganancia_private_estimada)
), by = .(modelo_id, envios)]

info_modelos <- mejores_modelos[, c(
  "ranking_origen",
  "id_origen",
  "metrica_origen",
  columnas_parametros
), with = FALSE]

info_modelos[, modelo_id := .I]

resumen_ganancias <- merge(
  info_modelos,
  resumen_metricas,
  by = "modelo_id"
)

setorder(
  resumen_ganancias,
  modelo_id,
  envios
)

resumen_ganancias[]


modelo_id,ranking_origen,id_origen,metrica_origen,learning_rate,num_iterations,num_leaves,min_sum_hessian_in_leaf,envios,ganancia_total_promedio,ganancia_total_sd,ganancia_public_promedio,ganancia_private_promedio
<int>,<int>,<chr>,<dbl>,<dbl>,<int>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,5,0.1182831,0.003697844,2780,24,0.06401048,4000,316800000,4115823.1,311410000,319110000
1,1,5,0.1182831,0.003697844,2780,24,0.06401048,4500,338470000,1967739.8,341953333,336977143
1,1,5,0.1182831,0.003697844,2780,24,0.06401048,5000,348920000,2847279.4,359755000,344276429
1,1,5,0.1182831,0.003697844,2780,24,0.06401048,5500,357390000,4441058.4,371525000,351332143
1,1,5,0.1182831,0.003697844,2780,24,0.06401048,6000,369160000,4441058.4,378693333,365074286
1,1,5,0.1182831,0.003697844,2780,24,0.06401048,6500,382910000,3697702.0,390573333,379625714
1,1,5,0.1182831,0.003697844,2780,24,0.06401048,7000,390720000,2951609.7,398878333,387223571
1,1,5,0.1182831,0.003697844,2780,24,0.06401048,7500,398750000,3300000.0,418806667,390154286
1,1,5,0.1182831,0.003697844,2780,24,0.06401048,8000,406340000,983869.9,433546667,394680000


In [29]:
mejor_corte_por_modelo <- resumen_ganancias[
  order(modelo_id, -ganancia_total_promedio),
  .SD[1L],
  by = modelo_id
]

setorder(mejor_corte_por_modelo, -ganancia_total_promedio)
mejor_corte_por_modelo[]


modelo_id,ranking_origen,id_origen,metrica_origen,learning_rate,num_iterations,num_leaves,min_sum_hessian_in_leaf,envios,ganancia_total_promedio,ganancia_total_sd,ganancia_public_promedio,ganancia_private_promedio
<int>,<int>,<chr>,<dbl>,<dbl>,<int>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,5,0.1182831,0.003697844,2780,24,0.06401048,12500,442530000,5355091,473275000,429353571
